# WP29 — Self-Play Synthesis Tournament
## Two Agents Compete and Teach Each Other via Mutual Distillation

---

This notebook demonstrates **WP29: Self-Play Synthesis Tournament**, which pairs two
`TransferCRLS` agents (Alpha and Beta) in a mutual teaching loop inspired by
AlphaGo Zero's self-play training paradigm.

### The Gap WP29 Closes

Every WP17–28 layer is self-contained: it learns from its own history. There is no
*external adversarial signal* — no other agent actively exposing the system's blind spots.

### Self-Play Mechanism

Each tournament match:
1. Both agents run on the **same puzzle batch** independently
2. **Winner teaches loser** via `teach_from_opponent()` — injects winner's synthesis action
   as a fine-tuning label into the loser's WP28 transfer student
3. **Winner's meta-params nudge loser** via `receive_tournament_signal()`
4. **Elo ratings** updated — convergence detected when Elo gap < threshold

### Theoretical Grounding
> *"The opponent is the only teacher that provably knows your current limitations."*
> — Silver et al., AlphaGo Zero (2017)

Runtime: **~10 min** (no GPU required)

In [ ]:
# ── 0. Environment setup ────────────────────────────────────────────────────
import sys, os, importlib

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
    print(f'Local mode — repo root: {repo_root}')

import warnings; warnings.filterwarnings('ignore')
import time, random, json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from prometheus.wp29_self_play_tournament import (
    SelfPlayTournament, TournamentCRLS, TournamentRecord,
    verify_wp29_exit_criteria,
)
from prometheus.wp18_analogy_engine import GoChessTacticRegistry
from prometheus.wp22_bandit_exploration import BanditMode
from prometheus.wp17_crls_synthesis import SynthesisAction
from prometheus.environments.go import GoBoard

import prometheus
print(f'Prometheus version: {prometheus.__version__}')
print('WP29 imports OK.')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 7), 'font.size': 11})
SEED = 42; random.seed(SEED); np.random.seed(SEED)

In [ ]:
# ── 1. Configuration ────────────────────────────────────────────────────────
QUICK_MODE      = True
N_MATCHES       = 8  if QUICK_MODE else 20
PUZZLES_PER_GEN = 25 if QUICK_MODE else 60
BOARD_SIZE      = 9
ELO_CONVERGENCE = 50.0   # Elo gap below which agents are considered equal

print(f'Mode: {"QUICK" if QUICK_MODE else "FULL"}')
print(f'Matches: {N_MATCHES} | Puzzles/match: {PUZZLES_PER_GEN}')
print(f'Elo convergence threshold: {ELO_CONVERGENCE}')

In [ ]:
# ── 2. Puzzle factory ────────────────────────────────────────────────────────
def make_atari_puzzle(board_size, rng):
    board = GoBoard(size=board_size)
    cx = board_size // 2
    stones = [(cx, cx), (cx, cx+1), (cx+1, cx)]
    for r, c in stones:
        if board.is_on_board(r, c) and board.board[r, c] == GoBoard.EMPTY:
            board.board[r, c] = GoBoard.BLACK
    all_libs = set()
    for r, c in stones:
        for nr, nc in board.get_neighbors(r, c):
            if board.board[nr, nc] == GoBoard.EMPTY:
                all_libs.add((nr, nc))
    libs = list(all_libs); rng.shuffle(libs)
    for r, c in libs[:-1]: board.board[r, c] = GoBoard.WHITE
    board.current_player = GoBoard.WHITE
    return board, GoBoard.WHITE, libs[-1]

def generate_puzzles(n, board_size, seed):
    rng = np.random.default_rng(seed)
    return [make_atari_puzzle(board_size, rng) for _ in range(n)]

def evaluate_move(board, move, correct_move, player):
    if move == correct_move: return True
    if board.is_legal_move(move[0], move[1], player):
        return len(board.would_capture(move[0], move[1], player)) > 0
    return False

print('Puzzle factory OK.')

---
## Section 1 — Instantiate Two TournamentCRLS Agents

Alpha and Beta are independent `TournamentCRLS` instances. They share no weights;
each has its own 10-layer synthesis stack. They will be played against each other
and teach each other via the self-play loop.

In [ ]:
# ── 3. Instantiate Alpha and Beta agents ─────────────────────────────────────
analogy_registry = GoChessTacticRegistry.build()
GO_STRATEGIES    = analogy_registry.go_strategies
CHESS_STRATEGIES = analogy_registry.chess_strategies
analogy_map      = analogy_registry.analogy_map

def make_agent(name, bandit_mode=BanditMode.UCB1):
    agent = TournamentCRLS(
        strategies                  = GO_STRATEGIES,
        bandit_mode                 = bandit_mode,
        analogy_map                 = analogy_map,
        source_strategies           = GO_STRATEGIES,
        target_strategies           = CHESS_STRATEGIES,
        transfer_min_finetune_steps = 3,
    )
    return agent

alpha = make_agent('Alpha', bandit_mode=BanditMode.UCB1)
beta  = make_agent('Beta',  bandit_mode=BanditMode.THOMPSON)  # different exploration strategy

tournament = SelfPlayTournament(
    alpha                      = alpha,
    beta                       = beta,
    initial_elo                = 1200.0,
    elo_convergence_threshold  = ELO_CONVERGENCE,
    convergence_patience       = 3,
    teach_alpha                = 0.10,
)

print('Tournament setup complete.')
print(f'  Alpha bandit: UCB1 | Beta bandit: Thompson')
print(f'  Initial Elo: 1200.0 (both)')
print(f'  Convergence: Elo gap < {ELO_CONVERGENCE} for 3 consecutive matches')

---
## Section 2 — Self-Play Tournament

In [ ]:
# ── 4. Run tournament ────────────────────────────────────────────────────────
print('=' * 80)
print(f'  Match  Alpha_acc  Beta_acc  Winner  Alpha_Elo  Beta_Elo  Elo_gap  Teach')
print('=' * 80)

for match_num in range(N_MATCHES):
    puzzles = generate_puzzles(PUZZLES_PER_GEN, BOARD_SIZE, seed=match_num * 251 + 13)

    # tactic_fns and evaluate_fn are passed to run_generation
    tactic_fns  = {}   # empty — strategies are internal
    evaluate_fn = evaluate_move

    rec = tournament.match(puzzles, tactic_fns, evaluate_fn)

    print(
        f'  {rec.match_number:5d}  {rec.alpha_accuracy:.3f}      {rec.beta_accuracy:.3f}     '
        f'{rec.winner:<6}  {rec.alpha_elo:.1f}     {rec.beta_elo:.1f}    '
        f'{rec.elo_gap:.1f}    '
        f'{"YES" if rec.loser_received_teach else "no"}'
    )
    if tournament.is_converged:
        print(f'\n  [Converged after {tournament.match_number} matches]')
        break

print('=' * 80)
summary = tournament.get_summary()
print(f'\nTournament summary:')
print(f'  Matches played: {summary["n_matches"]}')
print(f'  Alpha wins: {summary["alpha_wins"]} | Beta wins: {summary["beta_wins"]} | Draws: {summary["draws"]}')
print(f'  Final Elo — Alpha: {summary["final_alpha_elo"]:.1f} | Beta: {summary["final_beta_elo"]:.1f}')
print(f'  Converged: {summary["is_converged"]}')

In [ ]:
# ── 5. Visualisation ─────────────────────────────────────────────────────────
match_log = tournament.match_log
n_played  = len(match_log)
match_nums = [r.match_number for r in match_log]

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# A: Accuracy per agent per match
ax = axes[0, 0]
ax.plot(match_nums, [r.alpha_accuracy for r in match_log], 'b-o', linewidth=2.5,
        markersize=6, label='Alpha')
ax.plot(match_nums, [r.beta_accuracy  for r in match_log], 'r-s', linewidth=2.5,
        markersize=6, label='Beta')
ax.set_ylabel('Accuracy'); ax.set_title('Per-Match Accuracy: Alpha vs Beta', fontweight='bold')
ax.legend(fontsize=10); ax.set_ylim(0, 1.05)

# B: Elo ratings over time
ax2 = axes[0, 1]
ax2.plot(match_nums, [r.alpha_elo for r in match_log], 'b-o', linewidth=2.5,
         markersize=6, label='Alpha Elo')
ax2.plot(match_nums, [r.beta_elo  for r in match_log], 'r-s', linewidth=2.5,
         markersize=6, label='Beta Elo')
ax2.fill_between(match_nums,
                 [r.alpha_elo for r in match_log],
                 [r.beta_elo  for r in match_log],
                 alpha=0.15, color='purple', label='Elo gap')
ax2.axhline(1200, color='grey', linestyle='--', alpha=0.5, label='Initial Elo')
ax2.set_ylabel('Elo rating')
ax2.set_title('Elo Ratings During Tournament\n(convergence when gap < threshold)', fontweight='bold')
ax2.legend(fontsize=9)

# C: Elo gap over time with convergence threshold
ax3 = axes[1, 0]
elo_gaps = [r.elo_gap for r in match_log]
ax3.plot(match_nums, elo_gaps, 'purple', linewidth=2.5, marker='D', markersize=5)
ax3.axhline(ELO_CONVERGENCE, color='orange', linestyle='--', linewidth=2,
            label=f'Convergence threshold ({ELO_CONVERGENCE})')
ax3.fill_between(match_nums, 0, elo_gaps, alpha=0.2, color='purple')
ax3.set_xlabel('Match number'); ax3.set_ylabel('Elo gap |α_elo - β_elo|')
ax3.set_title('Elo Gap — Convergence Detection\n(self-play converges when agents equalise)', fontweight='bold')
ax3.legend(fontsize=10)

# D: Win distribution + summary
ax4 = axes[1, 1]
ax4.axis('off')
winner_counts = Counter(r.winner for r in match_log)
summary_text = (
    'WP29 Self-Play Tournament — Summary\n'
    '═════════════════════════════════════\n\n'
    f'  Matches played:    {summary["n_matches"]}\n'
    f'  Alpha wins:        {summary["alpha_wins"]} ({summary["alpha_win_rate"]:.1%})\n'
    f'  Beta wins:         {summary["beta_wins"]} ({summary["beta_win_rate"]:.1%})\n'
    f'  Draws:             {summary["draws"]} ({summary["draw_rate"]:.1%})\n\n'
    f'  Mean α accuracy:   {summary["mean_alpha_accuracy"]:.3f}\n'
    f'  Mean β accuracy:   {summary["mean_beta_accuracy"]:.3f}\n\n'
    f'  Final α Elo:       {summary["final_alpha_elo"]:.1f}\n'
    f'  Final β Elo:       {summary["final_beta_elo"]:.1f}\n'
    f'  Final Elo gap:     {summary["final_elo_gap"]:.1f}\n'
    f'  Converged:         {summary["is_converged"]}\n\n'
    'Silver et al. (2017):\n'
    '  Self-play as the sole\n'
    '  training signal; the\n'
    '  opponent IS the curriculum.\n\n'
    'Hofstadter (1979):\n'
    '  Agent A is simultaneously\n'
    '  the problem for B and the\n'
    '  meta-commentator on B.'
)
ax4.text(0.05, 0.95, summary_text, transform=ax4.transAxes,
         fontsize=9.5, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

fig.suptitle(
    'WP29: Self-Play Synthesis Tournament — Prometheus v0\n'
    'Mutual teaching via Elo-ranked tournament between two TransferCRLS agents',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('wp29_self_play_tournament.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to wp29_self_play_tournament.png')

In [ ]:
# ── 6. Verify WP29 exit criteria ─────────────────────────────────────────────
results = verify_wp29_exit_criteria(tournament)
print('WP29 Exit Criteria Verification')
print('=' * 50)
all_pass = True
for criterion, passed in results.items():
    status = '✓ PASS' if passed else '✗ FAIL'
    print(f'  {status}  {criterion}')
    if not passed: all_pass = False
print()
if all_pass:
    print('All WP29 exit criteria satisfied.')
    print('Self-play tournament is running correctly with mutual teaching.')
else:
    print('Some criteria not met — run more matches.')

---
## Conclusions

**Self-play** (Silver et al. 2017, Tesauro 1995) provides an adversarial signal that
no single-agent training loop can replicate: the opponent is the only teacher that
provably knows your current limitations. The `SelfPlayTournament` implements this
principle at the meta-policy level: two synthesis stacks teach each other how to
improve their improvement procedures.

The Elo convergence criterion (rather than a fixed number of matches) provides a
principled stopping condition: when both agents are equally matched, continued
self-play yields diminishing returns.

### References
- Silver, D. et al. (2017). Mastering the game of Go without human knowledge. *Nature*, 550, 354–359.
- Tesauro, G. (1995). Temporal difference learning and TD-Gammon. *CACM*, 38(3), 58–68.
- Hofstadter, D. (1979). *Gödel, Escher, Bach*. Basic Books.
- Good, I.J. (1965). Speculations concerning the first ultraintelligent machine.